# Scraping Jumia
This notebook covers the steps required for scrapnig jumia. It serves as an exploration facilitating understanding of the implementation of `JumiaScraper`

The first step is to choose an example product to work with

In [1]:
product = "laptop"

Then, we need to reach out through http and aquire the html content for a jumia search as though we are a typcical user

In [2]:
from requests import get as http_get

def jumia_search_url_for(product_name: str) -> str:
    return f"https://www.jumia.com.eg/catalog/?q={product_name.replace(' ', '+')}"

print(jumia_search_url_for(product))

res = http_get(jumia_search_url_for(product))
res.content

https://www.jumia.com.eg/catalog/?q=laptop


b'<!DOCTYPE html><html lang="en" dir="ltr"><head><meta charset="utf-8"/><title>Shop All Products @ #1 Online Store - Enjoy Online Shopping @ Best Prices - Jumia Egypt</title><meta property="og:locale" content="en_EG"/><meta property="og:locale:alternate" content="ar_EG"/><meta name="title" content="Shop All Products @ #1 Online Store - Enjoy Online Shopping @ Best Prices - Jumia Egypt"/><meta name="robots" content="noindex,follow"/><meta name="description" content="Select Your Favorite Products from Jumia Egypt\'s Online Store - Start Online Shopping Today for Products of All Kinds! - Fast Delivery - Free Returns"/><meta property="og:type" content="website"/><meta property="og:title" content="Shop All Products @ #1 Online Store - Enjoy Online Shopping @ Best Prices - Jumia Egypt"/><meta property="og:site_name" content="Jumia Egypt"/><meta property="og:description" content="Select Your Favorite Products from Jumia Egypt\'s Online Store - Start Online Shopping Today for Products of All K

The next step would be parsing the html. Through inspecting the page on a browser, each product is contained within an `article` tag within a `div` with the attirbute `data-catalog=true`.

In [3]:
from bs4 import BeautifulSoup, ResultSet

soup = BeautifulSoup(res.content, "html.parser")
product_cards: ResultSet = soup.select("[data-catalog=true] article")
product_cards

[<article class="prd _fb col c-prd"><a class="btn _i _rnd -mas -fsh0 -me-start _wslt _sec" data-ga4-discount="36.05" data-ga4-is_second_chance="false" data-ga4-item_brand="Asus" data-ga4-item_category="Computing" data-ga4-item_category2="Computers &amp; Accessories" data-ga4-item_category3="Computers &amp; Tablets" data-ga4-item_category4="Laptops" data-ga4-item_category5="Traditional Laptops" data-ga4-item_category6="Notebooks" data-ga4-item_id="AS040CL2YH3JNNAFAMZ" data-ga4-item_name="laptop E1504FA-NJ003W, AMD Ryzen 3 7320U, 256 GB SSD, 4 GB RAM , AMD Radeon Graphics,15.6 Inch FHD- " data-ga4-item_variant="" data-ga4-price="257.40" data-ga4-tags="CP_41" data-moengage-brand_key="asus" data-moengage-brand_name="Asus" data-moengage-category_key="notebooks" data-moengage-category_name="Notebooks" data-moengage-discount="36.05" data-moengage-item_variant="" data-moengage-product_image="https://eg.jumia.is/unsafe/fit-in/300x300/filters:fill(white)/product/78/8370431/1.jpg?9567" data-moeng

It is important to sccess the individual product pages for each listing, as they contain all the details we need. Right now, we have the html elements which represent each product on the search page selected. they each contain an anchor tag referencing the respective product page. This anchor tag is the second child of the tags we have selected and posess the class `core`. We will be selecting based upon the second feature, as this seems more stable. Lastly, the extracted linkes need to be suffixes for the root of the website `jumia.com`.

In [4]:
product_anchors = [product_card.select_one("a.core") for product_card in product_cards]
product_paths = [anchor['href'] for anchor in product_anchors]
product_links = [f"https://www.jumia.com.eg{path}" for path in product_paths]
product_links

['https://www.jumia.com.eg/asus-laptop-e1504fa-nj003w-amd-ryzen-3-7320u-256-gb-ssd-4-gb-ram-amd-radeon-graphics15.6-inch-fhd-windows-11-grey-green-134073887.html',
 'https://www.jumia.com.eg/asus-vivobook-go-15-e1504fa-bq823w-amd-ryzen-r3-7320u-amd-radeon-graphics-8gb-lpddr5-256gb-m.2-15.6-inch-fhd-60hz-win-11-green-gray.-134197947.html',
 'https://www.jumia.com.eg/lenovo-laptop-ip-slim-15iru8-ci3-1315u-8gb-256gb-ssd-intel-uhd-15.6-fhd-250nits-grey-82x700hrax-134041948.html',
 'https://www.jumia.com.eg/hp-fd0021nx-intel-core-i3-1315u-256gb-ssd-4gb-ram-intel-uhd-graphics-15.6-inch-fhd-silver-8f4t4ea-133919635.html',
 'https://www.jumia.com.eg/acer-5-a515-57g-56dq-slim-business-laptop-intel-core-i5-8gb-ram-512gb-ssd-130154272.html',
 'https://www.jumia.com.eg/lenovo-14e-chromebook-81mh000bus-14-chromebook-1920-x-1080-amd-a-series-a4-9120-dual-core-2-core-1.60-ghz-4-gb-ram-32-gb-flash-memory-133980592.html',
 'https://www.jumia.com.eg/lenovo-laptop-ideapad-3-slim-15irh10-i5-13420h-8g-512s

Each of these links needs to be visited and scraped. For the purpose of this investigation, only 5 links will be of studied. The procedure can be scaled to any number of links, though paralellism may be advised.

The first step is to fetch the page for the individual products

In [5]:
product_pages = [http_get(link) for link in product_links[:5]]
product_pages = [BeautifulSoup(page.content, "html.parser") for page in product_pages]
product_pages

[<!DOCTYPE html>
 <html dir="ltr" lang="en"><head><meta charset="utf-8"/><title>laptop E1504FA-NJ003W, AMD Ryzen 3 7320U, 256 GB SSD, 4 GB RAM , AMD Radeon Graphics,15.6 Inch FHD- Windows 11 – Grey Green - Asus - Jumia Egypt</title><meta content="en_EG" property="og:locale"/><meta content="ar_EG" property="og:locale:alternate"/><meta content="laptop E1504FA-NJ003W, AMD Ryzen 3 7320U, 256 GB SSD, 4 GB RAM , AMD Radeon Graphics,15.6 Inch FHD- Windows 11 – Grey Green - Asus - Jumia Egypt" name="title"/><meta content="index,follow" name="robots"/><meta content="Get Asus laptop E1504FA-NJ003W, AMD Ryzen 3 7320U, 256 GB SSD, 4 GB RAM , AMD Radeon Graphics,15.6 Inch FHD- Windows 11 – Grey Green online at Jumia and other Asus Notebooks at the best price in Egypt  ➤ Enjoy Amazing Offers &amp; Best Prices with Jumia Egypt - Free Returns - Cash on Delivery" name="description"/><meta content="product" property="og:type"/><meta content="laptop E1504FA-NJ003W, AMD Ryzen 3 7320U, 256 GB SSD, 4 GB RAM

Now, we need to extract the individual feature we are interested in from each prodcut page:

Title is the simplest: it is the text of the only `h1` element on the page.

In [6]:
titles = [page.select_one("h1").text for page in product_pages]
titles

['Asus laptop E1504FA-NJ003W, AMD Ryzen 3 7320U, 256 GB SSD, 4 GB RAM , AMD Radeon Graphics,15.6 Inch FHD- Windows 11 – Grey Green',
 'Asus Vivobook Go 15 E1504FA-BQ823W - AMD Ryzen R3 7320U - AMD Radeon Graphics - 8GB LPDDR5 - 256GB M.2 - 15.6-inch FHD 60Hz - Win 11 Green Gray.',
 'Lenovo LAPTOP IP Slim 15IRU8 Ci3-1315U, 8GB, 256GB SSD, Intel UHD, 15.6" FHD, 250nits, Grey 82X700HRAX ',
 'HP fd0021nx Intel Core i3-1315U 256GB SSD 4GB Ram Intel UHD Graphics 15.6 Inch FHD - Silver - 8F4T4EA',
 'Acer 5 A515-57G-56DQ Slim Business Laptop – Intel Core i5, 8GB RAM, 512GB SSD']

Price can be found using the regex `/EGP [0-9\|,\|.]+/` on stringified html

In [7]:
from typing import Optional
from re import search


def extract_price(html_str: str) -> Optional[float]:
    match = search(r"EGP [\d|,|\.]+", html_str)
    if match is not None:
        price_str = match.group().replace("EGP ", "").replace(",", "")
        return float(price_str)
    return None

prices = [extract_price(str(page)) for page in product_pages]
prices

[14979.0, 16609.0, 21244.0, 17649.0, 26666.0]

All images for a product are within `div#imgs`. I may select the first one. the `src` tag displays a wierd value, so i will use the `data-src` attribute present on the same element

In [8]:
images = [page.select_one("div#imgs img")["data-src"] for page in product_pages]

images

['https://eg.jumia.is/unsafe/fit-in/500x500/filters:fill(white)/product/78/8370431/1.jpg?9567',
 'https://eg.jumia.is/unsafe/fit-in/500x500/filters:fill(white)/product/74/9791431/1.jpg?5448',
 'https://eg.jumia.is/unsafe/fit-in/500x500/filters:fill(white)/product/84/9140431/1.jpg?1055',
 'https://eg.jumia.is/unsafe/fit-in/500x500/filters:fill(white)/product/53/6919331/1.jpg?8795',
 'https://eg.jumia.is/unsafe/fit-in/500x500/filters:fill(white)/product/27/2451031/1.jpg?3631']


Review count can be found by targetting the text within the anchor element with `href` prefixed `/catalog/productratingsreviews` and parsing out the number. In case there are no rating, there is no corresponding anchor on the page. In such a scenario, if the page containst the string `"(No ratings available)"`, then the review count can be safely assumed to be 0, otherwise, there is an unexpected scenario, hence the review count will be left unspecified.

In [9]:
def extract_review_count(page: BeautifulSoup) -> Optional[int]:
    anchors = page.select("a")
    try:
        reviews_anchor = [
            anchor
            for anchor in anchors
            if anchor["href"].startswith("/catalog/productratingsreviews")
        ][0]
    except IndexError:
        return 0 if "No ratings available" in page.text else None
    review_num_str = search(r"[\d|,]+", reviews_anchor.text).group().replace(",", "")
    return int(review_num_str)


review_counts = [extract_review_count(page) for page in product_pages]
review_counts

[0, 0, 0, 1, 0]

FInally, for ratings, they are displayed by tags which contain text formatted as "x out of 5" and a class of "stars". The actual rendering is done through a before pseuto-element. Any of these elements may be targetted and their text parsed to achieve the desired result

In [34]:
from bs4.element import Tag


def extract_rating(page: BeautifulSoup) -> Optional[float]:
    stars_element = page.select_one(".stars")
    if type(stars_element) != Tag:
        return None
    if not stars_element.text.endswith("out of 5"):
        return None
    rating_num_text = search(r"[\d.]+", stars_element.text).group(0)
    return float(rating_num_text) if rating_num_text != "0" else None

ratings = [extract_rating(page) for page in product_pages]
ratings

[None, None, None, 1.0, None]

Finally, the output can be displayed visually

In [36]:
from pandas import DataFrame

DataFrame(
    {
        "Names": titles,
        "Prices": prices,
        "Ratings": ratings,
        "Review count": review_counts,
        "Image URLs": images,
    }
)

,Names,Prices,Ratings,Review count,Image URLs
0,"Asus laptop E1504FA-NJ003W, AMD Ryzen 3 7320U,...",14979.0,NaN,0,https://eg.jumia.is/unsafe/fit-in/500x500/filt...
1,Asus Vivobook Go 15 E1504FA-BQ823W - AMD Ryzen...,16609.0,NaN,0,https://eg.jumia.is/unsafe/fit-in/500x500/filt...
2,"Lenovo LAPTOP IP Slim 15IRU8 Ci3-1315U, 8GB, 2...",21244.0,NaN,0,https://eg.jumia.is/unsafe/fit-in/500x500/filt...
3,HP fd0021nx Intel Core i3-1315U 256GB SSD 4GB ...,17649.0,1.0,1,https://eg.jumia.is/unsafe/fit-in/500x500/filt...
4,Acer 5 A515-57G-56DQ Slim Business Laptop – In...,26666.0,NaN,0,https://eg.jumia.is/unsafe/fit-in/500x500/filt...
